In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 2.3 Least Squares Four Ways

In [ ]:
from ecp.style import header, use_style

use_style()
header(
    volume="Chapter II — Orthogonality and Least Squares",
    number="2.3",
    title="Least Squares Four Ways",
    blurb="Four algorithms, one answer, and a factor of two million between "
    "them — because forming A-transpose-A squares the condition number, and at "
    "degree 14 it does not merely lose accuracy but fails outright.",
    difficulty="advanced",
    estimate="90–120 min",
)

## Notebook overview

[§2.1](projections-normal-equations.ipynb) derived the normal equations
$A^{\top}\!A\hat{\mathbf{x}} = A^{\top}\mathbf{b}$ and warned they are not how
least squares is solved in practice.
[§2.2](gram-schmidt-qr.ipynb) built the replacement. This notebook measures the
difference, and the difference is larger than the warning suggested.

The mechanism is one identity. For a matrix with independent columns,

$$
\kappa_2(A^{\top}\!A) = \kappa_2(A)^2 ,
$$

because the singular values of $A^{\top}A$ are the *squares* of those of $A$.
Forming the normal equations therefore takes a problem with condition number
$\kappa$ and hands the solver one with condition number $\kappa^2$, and by
[§0.2](../00-machine/floating-point.ipynb) that costs $\log_{10}\kappa$ extra
digits before any arithmetic happens.

Measured on polynomial fitting — where the design matrix is a Vandermonde
matrix and $\kappa$ grows fast with the degree — the consequences run:

| degree | $\kappa(A)$ | normal equations | $QR$ | factor |
|---|---|---|---|---|
| 3 | $10^{2}$ | $2.7\times10^{-13}$ | $4.2\times10^{-15}$ | 60 |
| 6 | $1.9\times10^{4}$ | $3.6\times10^{-9}$ | $1.3\times10^{-13}$ | 27,000 |
| 10 | $2.0\times10^{7}$ | $4.8\times10^{-4}$ | $2.2\times10^{-10}$ | 2,200,000 |
| 14 | $2.5\times10^{10}$ | **fails** | $2.3\times10^{-7}$ | — |

That last row is the one to remember. At degree 14 the normal equations do not
return a poor answer; the Cholesky factorization **breaks down**, because
$A^{\top}A$ is no longer numerically positive definite even though $A$ has full
column rank and the least-squares problem is perfectly well posed. The
information that made $A^{\top}A$ positive definite was destroyed in forming it.

The notebook then turns to what a fit *reports*. A least-squares answer without
an uncertainty is half an answer, and the covariance
$\sigma^2(A^{\top}A)^{-1}$ supplies it. We check those standard errors the only
honest way — by Monte Carlo, drawing four thousand noise realisations and
confirming the resulting $z$-scores really do have standard deviation 1.

> **How to read a check.** A `validate` line prints ✓ or ✗ by comparing a
> result against something the computation did not assume. A ✗ flags a
> mismatch to investigate, never a verdict on its own.

> **Scope.** Trefethen and Bau {cite}`trefethen1997` Lectures 11 and 18–19;
> Golub and Van Loan {cite}`golub2013` Chapter 5; the error analysis in full is
> Higham {cite}`higham2002` Chapter 20. For the statistical side, Hastie,
> Tibshirani and Friedman {cite}`hastie2009` Chapter 3.

## Theory in brief

### The squaring identity

If $A = U\Sigma V^{\top}$ then $A^{\top}A = V\Sigma^2V^{\top}$, so the singular
values square and

```{math}
:label: eq-ls-squared-kappa
\kappa_2(A^{\top}\!A) = \frac{\sigma_1^2}{\sigma_n^2} = \kappa_2(A)^2 .
```

This is not an artefact of any algorithm; it is a property of the matrix you
chose to build. Any method that forms $A^{\top}A$ inherits it.

### Four routes

All four solve $\min_{\mathbf{x}}\|A\mathbf{x} - \mathbf{b}\|_2$ and agree in
exact arithmetic.

1. **Normal equations with Cholesky.** Form $A^{\top}A$ (which is symmetric
   positive definite when $A$ has independent columns), factor it as
   $R^{\top}R$, and solve two triangular systems. Cost $\approx mn^2 + n^3/3$,
   the cheapest of the four. Accuracy $\kappa^2\varepsilon$.
2. **$QR$.** With $A = QR$ economic, the residual
   $\|A\mathbf{x} - \mathbf{b}\|$ equals $\|R\mathbf{x} - Q^{\top}\mathbf{b}\|$
   plus a term independent of $\mathbf{x}$, since $Q$ preserves norms. So

   ```{math}
   :label: eq-ls-qr-solve
   R\hat{\mathbf{x}} = Q^{\top}\mathbf{b} ,
   ```

   one triangular solve, no $A^{\top}A$ anywhere. Cost $\approx 2mn^2$,
   accuracy $\kappa\varepsilon$.
3. **SVD.** With $A = U\Sigma V^{\top}$ economic,
   $\hat{\mathbf{x}} = V\Sigma^{-1}U^{\top}\mathbf{b}$. Cost several times
   $QR$, accuracy $\kappa\varepsilon$, and — the reason it exists — it keeps
   working when the columns are *dependent*, which
   [§2.4](pseudoinverse-regularization.ipynb) needs and the other three cannot
   do.
4. **`np.linalg.lstsq`**, which is a driver around the SVD (LAPACK `gelsd`)
   with a rank tolerance.

The rule that follows: **use $QR$**, unless the matrix may be rank-deficient,
in which case use the SVD. Use the normal equations when $\kappa$ is small and
speed matters, knowing what you are spending.

### What a fit reports

If $\mathbf{b} = A\mathbf{x}_{\text{true}} + \boldsymbol{\epsilon}$ with
independent noise of variance $\sigma^2$, then $\hat{\mathbf{x}}$ is unbiased
with covariance

```{math}
:label: eq-ls-covariance
\operatorname{Cov}(\hat{\mathbf{x}}) = \sigma^2\,(A^{\top}\!A)^{-1},
\qquad
\hat{\sigma}^2 = \frac{\|\mathbf{b} - A\hat{\mathbf{x}}\|^2}{m - n} ,
```

the denominator $m-n$ being the residual degrees of freedom — the dimension of
$N(A^{\top})$ from [§1.4](../01-matrices/four-subspaces.ipynb), which is
exactly where the residual lives. The square roots of the diagonal are the
**standard errors**.

### Unequal uncertainties

When measurement $i$ has its own variance $\sigma_i^2$, weighting each equation
by $1/\sigma_i$ makes the noise uniform again. With $W = \operatorname{diag}(1/\sigma_i^2)$,

```{math}
:label: eq-ls-weighted
A^{\top}WA\,\hat{\mathbf{x}} = A^{\top}W\mathbf{b} ,
```

equivalently ordinary least squares on the row-scaled system
$\operatorname{diag}(1/\sigma_i)A$ and $\operatorname{diag}(1/\sigma_i)\mathbf{b}$
— which is how it should be computed, since that form never builds $A^{\top}WA$.

---
## Setup

Data and instruments only: the polynomial-fitting problem family (a given
specimen with a known exact answer) and an error meter. The four solution
routes are the exercises' work.

The Setup below holds this notebook's data and instruments — nothing you
are asked to build. It is collapsed so the building stays yours; expand it
whenever you want the details.

<!-- setup-policy: v2 -->

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import cho_factor, cho_solve, qr

from ecp import validate
from ecp import linalg as la
from ecp.style import use_style

use_style()
rng = np.random.default_rng(0)  # every random array below comes from this seed

EPS = np.finfo(float).eps
np.set_printoptions(precision=5, suppress=False, linewidth=110)


# data: the given problem family — a fitting problem with a known exact
# answer, handed to every exercise as its specimen.
def vandermonde_problem(degree, m=40):
    """A polynomial-fitting least-squares problem with a known exact answer.

    Builds the m-by-(degree+1) Vandermonde matrix of the points t = 0, ..., 1 and
    a right-hand side b = A x_true for a random x_true, so the exact solution is
    known and the error of any method can be measured rather than estimated.
    Vandermonde matrices are used because their condition number grows quickly
    and predictably with the degree, which is what makes the sweep in Exercise 3
    span ten orders of magnitude.

    Parameters
    ----------
    degree : int
        Polynomial degree; the matrix has ``degree + 1`` columns.
    m : int, default 40
        Number of sample points on [0, 1].

    Returns
    -------
    tuple
        ``(A, b, x_true, t)``.
    """
    t = np.linspace(0.0, 1.0, m)
    A = np.vander(t, degree + 1, increasing=True)
    x_true = np.random.default_rng(0).standard_normal(degree + 1)
    return A, A @ x_true, x_true, t


# instrument: an error meter against the known answer — reporting currency,
# not any exercise's lesson.
def relative_error(x, x_true):
    """Relative 2-norm error of a computed solution against the exact one."""
    return float(np.linalg.norm(x - x_true) / np.linalg.norm(x_true))

## Exercise 1: Measuring the squaring

{eq}`eq-ls-squared-kappa` is an identity, not an approximation, and it is worth
confirming numerically before relying on it — partly because it is easy to
verify, and partly because *where the verification breaks down* is itself
informative.

For Vandermonde matrices of degrees 3, 6, 10 and 14 on 40 points in $[0,1]$,
the condition numbers run from $10^{2}$ to $2.5\times10^{10}$. Taking
$\log_{10}$ of both sides of {eq}`eq-ls-squared-kappa`, the ratio
$\log_{10}\kappa(A^{\top}A) \big/ \log_{10}\kappa(A)$ should be exactly 2.

It comes out $2.0000$ at degrees 3, 6 and 10 — and $1.73$ at degree 14. The
identity has not failed; the *measurement* has. At degree 14,
$\kappa(A) = 2.5\times10^{10}$, so $\kappa(A^{\top}A)$ should be
$6\times10^{20}$ — beyond $1/\varepsilon$, which means `np.linalg.cond` cannot
resolve the smallest singular value of $A^{\top}A$ and returns $10^{18}$
instead. This is the same saturation caveat [§2.2](gram-schmidt-qr.ipynb) met
when fitting slopes, and it is worth naming twice: a quantity larger than
$1/\varepsilon$ cannot be measured in `float64`, so it should not be gated on.

**Part a)** For degrees 3, 6, 10 and 14, build the problem with
`vandermonde_problem(degree)` and report $\kappa(A)$ and $\kappa(A^{\top}A)$
using `np.linalg.cond`, together with the ratio of their base-10 logarithms.

**Part b)** Confirm {eq}`eq-ls-squared-kappa` where it can be measured:
check the ratio equals 2 to within $5\times10^{-3}$ for every degree with
$\kappa(A^{\top}A) < 1/\varepsilon$, and report which degree fails that
condition.

**Part c)** Confirm the mechanism rather than just the identity: compute the
singular values of $A$ and of $A^{\top}A$ with
`np.linalg.svd(..., compute_uv=False)` at degree 6, and compare against
$\sigma_i(A)^2$ **two ways**. Measured relative to the matrix scale
$\sigma_1^2$, the identity holds to $4\times10^{-16}$ — machine precision.
Measured *entrywise*, the relative error on the **smallest** squared singular
value is $4\times10^{-9}$, five orders worse, because that value is around
$10^{-7}$ and was computed from a matrix of condition number $\kappa^2$. The
damage is not spread evenly: forming $A^{\top}A$ ruins the small end
specifically, which is exactly the end least squares depends on. Report both
numbers, and confirm the entrywise error is concentrated on the last one or two
singular values.

In [ ]:
# (solution hidden on the public site)


### Validation 1

The identity is checked only where `float64` can measure it, and the degree
where it cannot is identified rather than quietly excluded. The singular-value
check confirms the *mechanism* — squaring — rather than only the consequence.

In [ ]:
validate.close(
    np.array(log_ratio)[measurable], np.full(int(measurable.sum()), 2.0),
    "kappa(A^T A) = kappa(A)^2 wherever it is measurable (Eq. 1)",
    rtol=0.0, atol=5e-3,
)
validate.check(
    not measurable[-1] and measurable[:-1].all(),
    "and at degree 14 it is not measurable: kappa(A^T A) would exceed 1/eps",
    f"kappa(A) = {kappa_A[-1]:.2e}, so kappa(A^T A) should be "
    f"{kappa_A[-1]**2:.2e} but cond returns {kappa_AtA[-1]:.2e}",
)
validate.close(
    s_AtA, s_A**2,
    "sigma(A^T A) = sigma(A)^2 to machine precision, scaled by the matrix",
    rtol=0.0, atol=1e-13 * s_A[0] ** 2,
)
validate.check(
    rel_entrywise[-1] > 1e4 * rel_entrywise[0],
    "but ENTRYWISE the damage lands on the smallest singular value",
    f"relative error {rel_entrywise[0]:.1e} on sigma_1^2 against "
    f"{rel_entrywise[-1]:.1e} on the smallest: squaring ruins the small end "
    "specifically, which is the end least squares depends on",
)
validate.check(
    rel_entrywise[-1] < np.linalg.cond(A6) ** 2 * EPS,
    "and that entrywise damage stays within the predicted kappa^2 * eps",
    f"{rel_entrywise[-1]:.2e} against {np.linalg.cond(A6)**2 * EPS:.2e}",
)
validate.check(
    all(a < b for a, b in zip(kappa_A, kappa_A[1:])),
    "and kappa(A) grows monotonically with the polynomial degree",
    f"{[f'{k:.1e}' for k in kappa_A]}: Vandermonde matrices are the standard "
    "badly-conditioned design matrix",
)

## Exercise 2: Four routes, one answer

Before measuring accuracy, establish that the four methods really do compute
the same thing, on a problem where all four work comfortably.

The $QR$ route deserves its derivation, because it is the one to use. Since $Q$
has orthonormal columns, $\|Q\mathbf{z}\| = \|\mathbf{z}\|$, so writing the
full $Q = [\,Q_1\ Q_2\,]$ and using $A = Q_1R$,

$$
\|A\mathbf{x} - \mathbf{b}\|^2
= \big\|Q^{\top}(A\mathbf{x} - \mathbf{b})\big\|^2
= \big\|R\mathbf{x} - Q_1^{\top}\mathbf{b}\big\|^2
  + \big\|Q_2^{\top}\mathbf{b}\big\|^2 .
$$

The second term does not involve $\mathbf{x}$ at all, so the minimum is
attained by killing the first, which is {eq}`eq-ls-qr-solve`. And the leftover
term is the squared residual: $\|Q_2^{\top}\mathbf{b}\|$ is exactly the
component of $\mathbf{b}$ in $N(A^{\top})$ that
[§1.4](../01-matrices/four-subspaces.ipynb) identified.

**Part a)** At degree 6, solve the problem four ways and report each solution:
normal equations via `scipy.linalg.cho_factor` and `cho_solve` on
`A.T @ A` and `A.T @ b`; $QR$ via `scipy.linalg.qr(A, mode="economic")` then
`np.linalg.solve(R, Q.T @ b)`; SVD via
`np.linalg.svd(A, full_matrices=False)` then
`Vt.T @ ((U.T @ b) / s)`; and `np.linalg.lstsq(A, b, rcond=None)[0]`.

**Part b)** Confirm all four agree with each other to a relative $10^{-8}$ at
this degree, and that each satisfies the normal equations residual
$\|A^{\top}(A\hat{\mathbf{x}} - \mathbf{b})\| \le 10^{-6}\|A^{\top}\mathbf{b}\|$
— the condition that *defines* a least-squares solution, checked independently
of how it was obtained.

**Part c)** Confirm the projection identity: for the $QR$ solution, check
$A\hat{\mathbf{x}} = Q Q^{\top}\mathbf{b}$ to $10^{-12}$, which says the fitted
values are the projection of $\mathbf{b}$ onto $C(A)$ — the same $P$ that
[§2.1](projections-normal-equations.ipynb) built from $A(A^{\top}A)^{-1}A^{\top}$,
now obtained without forming $A^{\top}A$.

In [ ]:
# (solution hidden on the public site)


### Validation 2

The normal-equation residual is the right independent check: it tests whether
each answer *is* a least-squares solution, using the defining condition
$A^{\top}(A\hat{\mathbf{x}} - \mathbf{b}) = \mathbf{0}$, without reference to
how it was computed or to the known $\mathbf{x}_{\text{true}}$.

In [ ]:
validate.check(
    pairwise < 1e-8,
    "all four methods agree to 1e-8 relative at degree 6",
    f"largest pairwise disagreement {pairwise:.2e}",
)
validate.check(
    max(ne_resid.values()) < 1e-6,
    "and each satisfies the defining condition A^T (A x - b) = 0",
    f"largest normal-equation residual {max(ne_resid.values()):.2e}, relative "
    "to ||A^T b||",
)
validate.close(
    fitted, projected,
    "A x_hat = Q Q^T b: the fitted values are the projection of b onto C(A)",
    rtol=0.0, atol=1e-12,
)
validate.close(
    np.linalg.norm(b6 - fitted), np.linalg.norm(b6 - Q6 @ (Q6.T @ b6)),
    "and the residual norm agrees with the orthogonal-complement component",
    rtol=0.0, atol=1e-12,
)

## Exercise 3: Where the routes separate, and where one stops working

Now the sweep. The four methods agreed at degree 6; raising the degree raises
$\kappa$, and {eq}`eq-ls-squared-kappa` says the normal equations should
degrade twice as fast in the exponent.

The measured accuracies:

| degree | $\kappa(A)$ | normal eqs | $QR$ | SVD | `lstsq` |
|---|---|---|---|---|---|
| 3 | $1.2\times10^{2}$ | $2.7\times10^{-13}$ | $4.2\times10^{-15}$ | $3.1\times10^{-15}$ | $4.8\times10^{-16}$ |
| 6 | $1.9\times10^{4}$ | $3.6\times10^{-9}$ | $1.3\times10^{-13}$ | $2.9\times10^{-14}$ | $1.3\times10^{-13}$ |
| 10 | $2.0\times10^{7}$ | $4.8\times10^{-4}$ | $2.2\times10^{-10}$ | $1.9\times10^{-10}$ | $1.1\times10^{-10}$ |
| 14 | $2.5\times10^{10}$ | **breaks down** | $2.3\times10^{-7}$ | $2.0\times10^{-7}$ | $2.2\times10^{-7}$ |

The three stable routes track $\kappa\varepsilon$; the normal equations track
$\kappa^2\varepsilon$. At degree 10 that is a factor of two million.

Degree 14 is qualitatively different and worth dwelling on. `cho_factor`
**raises an exception**: $A^{\top}A$ is not numerically positive definite, so
Cholesky — which is only defined for positive definite matrices — has nothing
to factor. But $A$ has full column rank, and the least-squares problem is
perfectly well posed; $QR$ solves it to seven digits. The positive-definiteness
was destroyed by the act of forming $A^{\top}A$, not by anything wrong with the
problem. That is as clean a demonstration as the subject offers that the
algorithm can be the whole difficulty.

**Part a)** For degrees 3, 6, 10 and 14, solve by all four routes, catching
`numpy.linalg.LinAlgError` from `cho_factor` and recording a failure rather
than crashing. Report the relative error of each against `x_true`, alongside
the reference values $\kappa\varepsilon$ and $\kappa^2\varepsilon$.

**Part b)** Confirm the separation: at degree 10 the normal-equation error must
exceed the $QR$ error by more than $10^{5}$, and the three stable routes must
all stay within a factor of 100 of $\kappa\varepsilon$.

**Part c)** Confirm the breakdown at degree 14: `cho_factor(A.T @ A)` must
raise, while `np.linalg.matrix_rank(A)` is still full and $QR$ returns an
answer with relative error below $10^{-5}$. Report the smallest eigenvalue of
$A^{\top}A$ from `np.linalg.eigvalsh` and confirm it has gone negative — the
numerical signature of lost positive definiteness. Plot all four error curves
against $\kappa(A)$ with the two reference lines.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 3

The degree-14 checks are the sharpest in the notebook: the Cholesky failure,
the negative eigenvalue that explains it, the full rank of $A$ that shows the
problem was fine, and the working $QR$ answer that proves it. Four statements
that together locate the failure entirely in the choice of algorithm.

In [ ]:
validate.check(
    err_normal[i10] / err_qr[i10] > 1e5,
    "at degree 10 the normal equations are >10^5 times less accurate than QR",
    f"{err_normal[i10]:.2e} against {err_qr[i10]:.2e}",
)
validate.check(
    all(e < 100 * k * EPS for e, k in zip(err_qr, kappas_sweep)),
    "QR stays within a factor of 100 of the kappa*eps limit at every degree",
    f"errors {[f'{e:.1e}' for e in err_qr]}",
)
validate.check(
    all(e < 100 * k * EPS for e, k in zip(err_svd, kappas_sweep))
    and all(e < 100 * k * EPS for e, k in zip(err_lstsq, kappas_sweep)),
    "and so do the SVD and lstsq: three stable routes, one unstable one",
    "the instability is specific to forming A^T A",
)
validate.check(
    chol_failed[14],
    "at degree 14 cho_factor RAISES: A^T A is not numerically positive definite",
    "not a poor answer -- no answer at all",
)
A_g14 = vandermonde_problem(14)[0]
gram_scale_14 = float(np.linalg.norm(A_g14.T @ A_g14, 2))
validate.check(
    min_eig[14] < 100 * EPS * gram_scale_14,
    "its smallest eigenvalue is numerically indistinguishable from zero",
    f"lambda_min(A^T A) = {min_eig[14]:.2e} against eps*||A^T A|| = "
    f"{EPS * gram_scale_14:.2e}: positive in exact arithmetic, and rounding "
    "is free to land it on either side of zero — only its smallness is gated",
)
validate.check(
    np.linalg.matrix_rank(vandermonde_problem(14)[0]) == 15 and err_qr[-1] < 1e-5,
    "while A has full column rank and QR solves the problem to 7 digits",
    f"rank 15 of 15, QR error {err_qr[-1]:.2e}: the problem was never the issue",
)

## Exercise 4: What the fit reports: residuals and standard errors

A fitted parameter without an uncertainty is not a result. {eq}`eq-ls-covariance`
supplies one, and this exercise checks it the only way that really counts.

The usual practice is to compute $\hat{\sigma}^2(A^{\top}A)^{-1}$, take square
roots of the diagonal, and quote them. But that formula rests on assumptions —
independent noise, correct model, the right degrees-of-freedom denominator —
and quoting a standard error one has not verified is quoting a number one does
not know to be calibrated. The verification is a Monte Carlo: draw many noise
realisations, refit each, and check that the **$z$-scores**
$(\hat{x}_j - x_{\text{true},j})/\mathrm{SE}_j$ really do have standard
deviation 1. If the standard errors are right, they will; if the formula is
being misapplied, they will not.

The data: $m = 60$ points on $[0,1]$, $y = 2 + 3t + \epsilon$ with
$\epsilon \sim \mathcal{N}(0, 0.15^2)$. The denominator in
{eq}`eq-ls-covariance` is $m - n = 58$, and that $58$ is not arbitrary — it is
$\dim N(A^{\top})$ from [§1.4](../01-matrices/four-subspaces.ipynb), the
dimension of the space the residual lives in. Dividing by $m$ instead would
bias $\hat{\sigma}^2$ low, because the residual has already been made as small
as two free parameters can make it.

**Part a)** Generate the data with `rng.standard_normal(60)` scaled by
$\sigma = 0.15$, fit by $QR$, and report $\hat{\mathbf{x}}$, the residual norm,
the RMSE $\|\mathbf{r}\|/\sqrt{m}$, and
$\hat{\sigma}^2 = \|\mathbf{r}\|^2/(m-2)$ against the true $\sigma^2 = 0.0225$.

**Part b)** Compute the covariance from {eq}`eq-ls-covariance` as
`sigma2_hat * np.linalg.inv(A.T @ A)` and the standard errors as the square
roots of its diagonal. Confirm the true parameters $(2, 3)$ lie within three
standard errors of the fit, and report the $z$-scores.

**Part c)** Calibrate the standard errors by Monte Carlo: draw 4000 fresh noise
realisations, refit each by $QR$, recompute its own standard errors, and
collect the $z$-scores. Confirm their standard deviation is $1.00 \pm 0.15$ and
their mean $0.00 \pm 0.15$ for both parameters, and that the fraction with
$|z| < 3$ is $0.997 \pm 0.01$ — the standard errors are then verified rather
than merely computed.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 4

The Monte Carlo is the validation that matters: it tests the standard errors
against their operational meaning — that a $z$-score is standard normal — over
4000 independent realisations. Computing $\sqrt{\operatorname{diag}(\hat{\sigma}^2
(A^{\top}A)^{-1})}$ and quoting it would test nothing at all.

In [ ]:
validate.check(
    bool(np.all(np.abs(z_scores) < 3.0)),
    "the true parameters (2, 3) lie within three standard errors of the fit",
    f"z-scores {np.round(z_scores, 3)}",
)
validate.close(
    sigma2_hat, SIGMA_TRUE**2,
    "the variance estimate with m - n degrees of freedom recovers sigma^2",
    # one realisation of a chi-square with m - n = 58 dof: sd(s^2/sigma^2) =
    # sqrt(2/58) = 19%, and this draw sits at one sd of it. Gate at ~4 sd.
    rtol=0.75, atol=0.0,
)
validate.close(
    z_mc.std(axis=0), np.ones(2),
    "over 4000 refits the z-scores have standard deviation 1: the SEs are calibrated",
    rtol=0.0, atol=0.15,
)
validate.close(
    z_mc.mean(axis=0), np.zeros(2),
    "and mean zero, so the estimator is unbiased", rtol=0.0, atol=0.15,
)
validate.close(
    np.mean(np.abs(z_mc) < 3, axis=0), np.full(2, 0.9973),
    "with 99.7% of refits inside three standard errors, as a normal requires",
    rtol=0.0, atol=0.01,
)

## Exercise 5: When measurements are not equally trustworthy

Ordinary least squares gives every equation the same say. That is correct only
when every measurement has the same uncertainty, and it is wrong — sometimes
badly — when they do not.

The failure mode is easy to see. If half the data has $\sigma = 0.05$ and half
has $\sigma = 0.5$, then the noisy half contributes a hundred times more to
$\sum r_i^2$ per unit of parameter error, so minimising the unweighted sum
lets the unreliable measurements dominate the answer. Weighting each equation
by $1/\sigma_i$ makes the noise uniform again and restores the ordinary
geometry.

{eq}`eq-ls-weighted` writes this as $A^{\top}WA\hat{\mathbf{x}} =
A^{\top}W\mathbf{b}$, but that form should not be *computed*: it forms a
weighted version of $A^{\top}A$ and inherits everything Exercise 3 measured.
The right implementation scales the rows and runs ordinary $QR$ on the result,
which is both stabler and simpler.

**Part a)** Build the data: $m = 60$ points on $[0,1]$, $y = 2 + 3t + \epsilon_i$
with $\sigma_i = 0.05$ for $t_i < 0.5$ and $\sigma_i = 0.5$ otherwise, using
`np.where(t < 0.5, 0.05, 0.5)` for the noise scale.

**Part b)** Fit twice: ordinary least squares with
`np.linalg.lstsq(A, b, rcond=None)[0]`, and weighted least squares by row
scaling — form `s = 1 / sigma_i`, then solve
`np.linalg.lstsq(s[:, None] * A, s * b, rcond=None)[0]`. Report both parameter
errors against the true $(2, 3)$ and confirm the weighted fit is at least five
times more accurate.

**Part c)** Confirm the row-scaling route is the same computation as
{eq}`eq-ls-weighted`: solve $A^{\top}WA\hat{\mathbf{x}} = A^{\top}W\mathbf{b}$
directly with `np.linalg.solve` and `W = np.diag(1 / sigma_i**2)`, and check the
two agree to $10^{-10}$ — the same answer, obtained without forming
$A^{\top}WA$.

```{admonition} With your assistant
:class: tip
Ask for `total_least_squares(A, b)`, which minimises perpendicular distance
rather than vertical distance — the right fit when the *inputs* $t_i$ carry
error too, and a genuinely different problem solved by an SVD of the augmented
matrix $[\,A\ \ \mathbf{b}\,]$. Then check it yourself: on data with noise
added to both $t$ and $y$ it must recover the true slope more accurately than
ordinary least squares over 200 trials, it must reduce exactly to ordinary
least squares when the input noise is zero (to $10^{-8}$), and its residual
must be smaller than OLS's when measured perpendicularly and larger when
measured vertically. The check is yours.
```

In [ ]:
# (solution hidden on the public site)


### Validation 5

The two routes to the weighted solution are required to agree, which shows the
row-scaling implementation is not an approximation but the same computation
arranged to avoid forming $A^{\top}WA$ — the lesson of Exercise 3 applied to a
new problem.

In [ ]:
validate.check(
    err_ols / err_wls > 5.0,
    "weighting by 1/sigma_i improves the fit by more than 5x",
    f"OLS error {err_ols:.4f} against WLS {err_wls:.4f}",
)
validate.close(
    x_wls, x_wls_normal,
    "row scaling and the A^T W A form of Eq. 4 give the same answer",
    rtol=0.0, atol=1e-10,
)
validate.check(
    err_wls < 0.1,
    "and the weighted fit recovers the true (2, 3) to better than 0.1",
    f"fitted {np.round(x_wls, 4)}",
)
validate.close(
    np.linalg.lstsq(np.ones(M_STAT)[:, None] * A_w,
                    np.ones(M_STAT) * b_w, rcond=None)[0],
    x_ols,
    "with equal weights the weighted fit reduces to ordinary least squares",
    rtol=0.0, atol=1e-12,
)

## Notebook summary

One problem, four algorithms, and a factor of two million.

The concrete results:

- $\kappa(A^{\top}A) = \kappa(A)^2$ held to $10^{-3}$ in the log ratio at every
  degree where `float64` can measure it, verified at the level of the
  mechanism — the singular values of $A^{\top}A$ are the squares of those of
  $A$ to $4\times10^{-16}$ **relative to the matrix scale** — while entrywise
  the relative error on the *smallest* squared singular value is
  $4\times10^{-9}$, five orders worse, because squaring ruins the small end
  specifically. And the identity is *not* measurable at degree 14, where the
  true $\kappa(A^{\top}A) \approx 6\times10^{20}$ exceeds $1/\varepsilon$;
- all four routes agreed to $10^{-8}$ at degree 6 and each satisfied the
  defining condition $A^{\top}(A\hat{\mathbf{x}} - \mathbf{b}) = \mathbf{0}$,
  with $A\hat{\mathbf{x}} = QQ^{\top}\mathbf{b}$ confirming the fitted values
  are [§2.1](projections-normal-equations.ipynb)'s projection;
- across degrees 3 to 14 the three stable routes tracked $\kappa\varepsilon$
  while the normal equations tracked $\kappa^2\varepsilon$, a gap of
  $2\times10^{6}$ at degree 10;
- and at degree 14 the normal equations did not degrade but **broke down**:
  `cho_factor` raised, the smallest eigenvalue of $A^{\top}A$ having gone
  **negative**, while $A$ still had full column rank 15 and $QR$ returned seven
  correct digits. The positive definiteness was destroyed by forming
  $A^{\top}A$, not by anything wrong with the problem;
- the standard errors from $\hat{\sigma}^2(A^{\top}A)^{-1}$ were **calibrated
  by Monte Carlo** over 4000 noise realisations: the $z$-scores came out with
  standard deviation $1.03$, mean $0.03$, and $99.6\%$ inside three standard
  errors against the $99.73\%$ a normal requires;
- and weighting by $1/\sigma_i$ on heteroscedastic data improved the parameter
  error by more than $12\times$, with the row-scaling implementation agreeing
  with the $A^{\top}WA$ form to $10^{-10}$ while never building it.

Methods met: `scipy.linalg.cho_factor`/`cho_solve`, $QR$ least squares via
{eq}`eq-ls-qr-solve`, the SVD route, `np.linalg.lstsq`, the covariance formula
{eq}`eq-ls-covariance` with $m-n$ degrees of freedom, Monte Carlo calibration
of standard errors, and weighted least squares by row scaling.

## Outlook

- **When the columns are dependent.** Every method here assumed $A$ has
  independent columns; $A^{\top}A$ is then invertible and $R$ nonsingular. Drop
  that and three of the four routes fail outright while the SVD keeps working,
  returning the minimum-norm solution.
  [§2.4](pseudoinverse-regularization.ipynb) builds it.
- **Choosing not to fit exactly.** At degree 14 even $QR$ gave only seven
  digits, because the problem itself is that ill-conditioned. When the data is
  noisy the right response is often to *deliberately* bias the fit —
  regularization — which trades a little accuracy for a lot of stability, and
  is [§2.4](pseudoinverse-regularization.ipynb) again.
- **A better basis.** The Vandermonde matrix was badly conditioned because the
  monomials are a poor basis, exactly as
  [§1.5](../01-matrices/vector-spaces-coordinates.ipynb) measured. Fitting in
  an orthogonal polynomial basis makes $A^{\top}A$ nearly diagonal and the
  whole difficulty evaporates — [§2.5](function-space-bases.ipynb) constructs
  one.
- **Least squares as learning.** Fitting parameters to data by minimising
  squared error is the simplest supervised learning there is, and the
  $\kappa$ story reappears there as the reason ridge regression exists.
  [§8.1](../08-learning/learning-as-least-squares.ipynb) picks it up, with the
  SVD filter that makes the connection precise.

### References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()